# Cyclone Mocha — ERA5 Pressure Field + IBTrACS Track

Combines:
- **ERA5 MSL pressure** (hourly, Copernicus CDS) with a time slider
- **IBTrACS best-track** positions for Cyclone Mocha (NOAA, NI basin)

At each timestep the ERA5 pressure field is shown and all IBTrACS track points
up to that time are plotted, building the full track as the slider advances.

> **Prerequisites**
> - `~/.cdsapirc` with your Copernicus CDS API key
> - ERA5 files downloaded by `myanmar_cyclone_mocha_may2023.ipynb`
>   (or re-downloaded below)

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import hvplot.xarray
import hvplot.pandas
import geoviews.tile_sources as gvts
import holoviews as hv
import panel as pn
from holoviews import opts

hv.extension('bokeh')
pn.extension()

print('Setup complete')

## 2. Load ERA5 MSL pressure

In [ ]:
from pathlib import Path
import cdsapi

ERA5_MSL_FILE = Path('era5_mocha_msl_1h.nc')

AREA  = [21.3, 90.9, 17.5, 94.6]   # [N, W, S, E]

DATES = [f'2023-05-{d:02d}' for d in range(12, 16)]
HOURS = [f'{h:02d}:00' for h in range(24)]

if not ERA5_MSL_FILE.exists():
    print('Downloading ERA5 MSL pressure ...')
    c = cdsapi.Client()
    c.retrieve(
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable':     ['mean_sea_level_pressure'],
            'date':         DATES,
            'time':         HOURS,
            'area':         AREA,
            'format':       'netcdf',
        },
        str(ERA5_MSL_FILE),
    )
else:
    print(f'Using cached {ERA5_MSL_FILE}')

ds_msl = xr.open_dataset(ERA5_MSL_FILE)

msl_var  = next(v for v in ds_msl.data_vars if 'msl' in v.lower() or 'pressure' in v.lower())
lat_dim  = 'latitude'   if 'latitude'   in ds_msl.dims else 'lat'
lon_dim  = 'longitude'  if 'longitude'  in ds_msl.dims else 'lon'
time_dim = 'valid_time' if 'valid_time' in ds_msl.dims else 'time'

msl_hpa = ds_msl[msl_var] / 100  # Pa → hPa
print(ds_msl)

## 3. Load IBTrACS best-track for Cyclone Mocha

In [ ]:
IBTRACS_URL = (
    'https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs'
    '/v04r01/access/csv/ibtracs.NI.list.v04r01.csv'
)

ibt = pd.read_csv(IBTRACS_URL, skiprows=[1], low_memory=False)

mocha = (
    ibt[
        (ibt['NAME'].str.upper() == 'MOCHA') &
        (ibt['SEASON'].astype(str) == '2023')
    ]
    .copy()
)

mocha['time'] = pd.to_datetime(mocha['ISO_TIME'])
mocha['lat']  = pd.to_numeric(mocha['LAT'],      errors='coerce')
mocha['lon']  = pd.to_numeric(mocha['LON'],      errors='coerce')
mocha['pres'] = pd.to_numeric(mocha['WMO_PRES'], errors='coerce')
mocha['wind'] = pd.to_numeric(mocha['WMO_WIND'], errors='coerce')
mocha = mocha.dropna(subset=['lat', 'lon']).reset_index(drop=True)

T_START = pd.Timestamp('2023-05-13 12:00')
T_END   = pd.Timestamp('2023-05-14 12:00')

mocha = mocha[(mocha['time'] >= T_START) & (mocha['time'] <= T_END)].reset_index(drop=True)

print(f'IBTrACS Mocha track points in window: {len(mocha)}')
print(f'Period: {mocha["time"].iloc[0]}  →  {mocha["time"].iloc[-1]}')
mocha[['time', 'lat', 'lon', 'pres', 'wind']].head(10)

## 4. Interactive plot — pressure field + cumulative IBTrACS track

In [ ]:
times = ds_msl[time_dim].values
times = times[(times >= np.datetime64(T_START)) & (times <= np.datetime64(T_END))]

time_slider = pn.widgets.DiscreteSlider(
    name='Time (UTC)',
    options={str(pd.Timestamp(t)): t for t in times},
    width=700,
)

@pn.depends(time_slider)
def cyclone_plot(t):
    t_ts = pd.Timestamp(t)

    pressure = msl_hpa.sel({time_dim: t}).hvplot.contourf(
        x=lon_dim, y=lat_dim,
        geo=True,
        levels=20,
        cmap='RdBu_r',
        clim=(960, 1015),
        alpha=0.6,
        colorbar=True,
        clabel='MSL pressure (hPa)',
        tiles='EsriImagery',
    )

    track_so_far = mocha[mocha['time'] <= t_ts]

    layers = pressure
    if not track_so_far.empty:
        layers = layers * track_so_far.hvplot.points(
            x='lon', y='lat',
            geo=True,
            c='wind',
            cmap='Greys',
            clim=(20, 120),
            size=60,
            colorbar=True,
            clabel='Wind speed (kt, IBTrACS)',
            hover_cols=['time', 'pres', 'wind'],
        )

    return layers.opts(
        opts.Points(tools=['hover']),
        opts.Overlay(
            width=750, height=600,
            title=f'Cyclone Mocha — {t_ts.strftime("%Y-%m-%d %H:%M UTC")}',
        ),
    )

pn.Column(time_slider, cyclone_plot)